### **Scenario:**
You're a data engineer intern at a growing e-commerce startup. The analytics team handed you a CSV file with **8 million transaction records** from the past year.  

Unfortunately... it’s a mess:
- The data is inconsistent
- Column types are incorrect
- There are missing values, duplicates, mixed date formats
- And it's eating **way too much memory**

Your mission is to clean it up — both **locally with Pandas** and **at scale using PySpark** — while keeping everything **memory-efficient and performant**.

---

### **Dataset Preview:**
The file is called `transactions.csv` (4 million rows) and contains:

| Column             | Description                                         |
|--------------------|-----------------------------------------------------|
| `customer_id`      | ID of the customer (some are missing)              |
| `transaction_id`   | Unique transaction string like `TXN1234567`        |
| `purchase_amount`  | Sometimes a float, sometimes a string, sometimes blank |
| `currency`         | Should be all 'USD', but has lowercase/missing     |
| `purchase_date`    | Mixed formats like `'2023/01/01'`, `'01-02-2023'`  |
| `product_id`       | Product ID, might be null                          |
| `product_category` | Category like 'Electronics', 'Books', messy casing |
| `is_returned`      | Values like `'yes'`, `True`, `'no'`, `False`, NaN  |

---

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('DE/transactions.csv', nrows = 1000000)


In [3]:
df.memory_usage(deep=True).sum() / 1024**2  # in MB

np.float64(312.41750049591064)

In [3]:
df.head(5)

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794.0,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
1,10859.0,TXN1419610,NaN,usd,2023/01/01,P_234,Books,no
2,86819.0,TXN5614226,-15.00,USD,2023/01/01,P_456,toys,True
3,64885.0,TXN5108603,-15.00,USD,03.04.2023,P_234,Books,True
4,16264.0,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True


In [4]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   customer_id       999992 non-null   float64
 1   transaction_id    1000000 non-null  object 
 2   purchase_amount   714117 non-null   float64
 3   currency          666363 non-null   object 
 4   purchase_date     800214 non-null   object 
 5   product_id        799878 non-null   object 
 6   product_category  1000000 non-null  object 
 7   is_returned       800016 non-null   object 
dtypes: float64(2), object(6)
memory usage: 312.4 MB


The 100,000 rows of data is using 61 ~ 312 MB of memory. That is very inefficient like the use case scenario stated at the beginning. There are a lot of null values in some columns which need to be dropped. 

I will proceed to check memory usage per columns to uncover which columns are using the memory more

In [19]:
cols = ['currency', 'customer_id', 'transaction_id', 'purchase_amount', 'purchase_date', 'product_id', 'product_category', 'is_returned']

for col in cols:
    print("\n", df[col].info())


<class 'pandas.core.series.Series'>
RangeIndex: 1000000 entries, 0 to 999999
Series name: currency
Non-Null Count   Dtype 
--------------   ----- 
666363 non-null  object
dtypes: object(1)
memory usage: 7.6+ MB

 None
<class 'pandas.core.series.Series'>
RangeIndex: 1000000 entries, 0 to 999999
Series name: customer_id
Non-Null Count   Dtype  
--------------   -----  
999992 non-null  float64
dtypes: float64(1)
memory usage: 7.6 MB

 None
<class 'pandas.core.series.Series'>
RangeIndex: 1000000 entries, 0 to 999999
Series name: transaction_id
Non-Null Count    Dtype 
--------------    ----- 
1000000 non-null  object
dtypes: object(1)
memory usage: 7.6+ MB

 None
<class 'pandas.core.series.Series'>
RangeIndex: 1000000 entries, 0 to 999999
Series name: purchase_amount
Non-Null Count   Dtype  
--------------   -----  
714117 non-null  float64
dtypes: float64(1)
memory usage: 7.6 MB

 None
<class 'pandas.core.series.Series'>
RangeIndex: 1000000 entries, 0 to 999999
Series name: purchase_date

Looks like the memory usage is shared accordingly accross all the columns. No column has a higher usage, with all using 7.6+

In [5]:
#drop duplicates
df.drop_duplicates()


,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794.0,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
1,10859.0,TXN1419610,NaN,usd,2023/01/01,P_234,Books,no
2,86819.0,TXN5614226,-15.00,USD,2023/01/01,P_456,toys,True
3,64885.0,TXN5108603,-15.00,USD,03.04.2023,P_234,Books,True
4,16264.0,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True
...,...,...,...,...,...,...,...,...
999995,59144.0,TXN9767846,60.00,usd,2023/01/01,NaN,toys,no
999996,74965.0,TXN1873282,120.50,NaN,2023/01/01,P_123,toys,NaN
999997,51203.0,TXN6732173,NaN,usd,03.04.2023,P_123,toys,False
999998,92591.0,TXN6197631,NaN,NaN,01-02-2023,P_456,books,False


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [7]:
# drop nulls in purchase amount
df.dropna(subset=['purchase_amount'], inplace=True)

In [10]:
# View rows with more than 2 missing values
cols = ['transaction_id', 'purchase_amount', 'purchase_date']

#filter
rows_with_missing = df[df[cols].isnull().sum(axis=1) < 3]

In [11]:
rows_with_missing.head(5)

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794.0,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
2,86819.0,TXN5614226,-15.00,USD,2023/01/01,P_456,toys,True
3,64885.0,TXN5108603,-15.00,USD,03.04.2023,P_234,Books,True
4,16264.0,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True
5,92385.0,TXN3341057,120.50,USD,NaN,P_234,TOYS,True


In [8]:
# drop rows with nulls in specified columns
df.drop(df[df['purchase_amount'].isna() & df['purchase_date'].isna() & df['transaction_id'].isna()].index, inplace=True)

In [9]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 714117 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   customer_id       714111 non-null  float64
 1   transaction_id    714117 non-null  object 
 2   purchase_amount   714117 non-null  float64
 3   currency          475755 non-null  object 
 4   purchase_date     571449 non-null  object 
 5   product_id        571211 non-null  object 
 6   product_category  714117 non-null  object 
 7   is_returned       571170 non-null  object 
dtypes: float64(2), object(6)
memory usage: 228.5 MB



I dropped rows with null values in all three columns which are needed for business insights (['transaction_id', 'purchase_amount', 'purchase_date']).

The memory usage reduced from the 300+ MB to 228.5MB.

In [ ]:
# convert customer_id, transaction_id, product_id, and product_category to string
#df = df.astype({
#    'transaction_id': 'str',
#    'product_id': 'str',
#    'product_category': 'str'
#})

In [10]:
# Convert customer if to category type
df['customer_id'] = df['customer_id'].astype('Int32').astype('category')

In [32]:
# convert transaction_id to string
df['transaction_id'] = df['transaction_id'].astype('string')

In [12]:
# Clean purchase_amount column before type conversion
df['purchase_date'] = df['purchase_date'].str.strip().str.replace('.','/').str.replace('-', '/')

In [13]:
from datetime import datetime

def parse_date(value):
    for fmt in ("%Y/%m/%d", "%/m/%d/%Y", "%d/%m/%Y"):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    return pd.NaT  # Return Not a Time if no format matches

In [14]:
# apply date parsing

df['parsed_date'] = df['purchase_date'].astype(str).apply(parse_date)

In [15]:
# drop purchase_date column
df.drop(columns=['purchase_date'], inplace=True)

# rename parsed_date to purchase_date
df.rename(columns={'parsed_date': 'purchase_date'}, inplace=True)

# reorder columns
df = df[['customer_id', 'transaction_id', 'product_id', 'purchase_amount', 'purchase_date', 
         'product_category', 'currency', 'is_returned']]

In [16]:
df.head(5)


,customer_id,transaction_id,product_id,purchase_amount,purchase_date,product_category,currency,is_returned
0,25794,TXN2867825,P_234,120.50,2023-03-15,books,USD,no
2,86819,TXN5614226,P_456,-15.00,2023-01-01,toys,USD,True
3,64885,TXN5108603,P_234,-15.00,2023-04-03,Books,USD,True
4,16264,TXN4744854,P_456,45.99,2023-04-03,Electronics,USD,True
5,92385,TXN3341057,P_234,120.50,NaT,TOYS,USD,True


In [17]:
# standardize currency columns
df['currency'].unique()

df['currency'] = df['currency'].astype('str').str.strip().str.upper()

In [18]:
df['is_returned'].unique()

# standardize the is_returned text to lowercase and strip whitespace
# create a new column 'returned' for easier analysis
df['returned'] = df['is_returned'].astype(str).str.strip().str.lower()

In [19]:
# map 'no' to False and 'yes' to True

mapping = {
    'no':'No',
    'yes': 'Yes',
    'False': 'No',
    'True': 'Yes'}

df['returned'] = df['is_returned'].map(mapping)

In [20]:
# convert 'returned' to boolean
df['_returned'] = df['returned'].map({'Yes': True, 'No': False})

In [21]:
df.drop(columns=['is_returned', 'returned'], inplace=True)
df.rename(columns={'_returned': 'is_returned'}, inplace=True)

In [22]:
df['is_returned'] = df['is_returned'].astype(bool)

In [23]:
# convert currency to category datatype
df['currency'] = df['currency'].astype('category')

In [24]:
# standardize product_category
df['product_category'] = df['product_category'].astype(str).str.strip().str.lower()

In [28]:
# convert product_category to category datatype
df['product_category'] = df['product_category'].astype('category')

In [25]:
# convert product_id to category type
df['product_id'] = df['product_id'].astype('category')

In [33]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 714117 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   customer_id       714111 non-null  category      
 1   transaction_id    714117 non-null  string        
 2   product_id        571211 non-null  category      
 3   purchase_amount   714117 non-null  float64       
 4   purchase_date     571449 non-null  datetime64[ns]
 5   product_category  714117 non-null  category      
 6   currency          714117 non-null  category      
 7   is_returned       714117 non-null  bool          
dtypes: bool(1), category(4), datetime64[ns](1), float64(1), string(1)
memory usage: 63.9 MB


In [35]:
df.head(10)

,customer_id,transaction_id,product_id,purchase_amount,purchase_date,product_category,currency,is_returned
0,25794,TXN2867825,P_234,120.50,2023-03-15,books,USD,False
2,86819,TXN5614226,P_456,-15.00,2023-01-01,toys,USD,True
3,64885,TXN5108603,P_234,-15.00,2023-04-03,books,USD,True
4,16264,TXN4744854,P_456,45.99,2023-04-03,electronics,USD,True
5,92385,TXN3341057,P_234,120.50,NaT,toys,USD,True
7,97497,TXN2458591,NaN,45.99,2023-01-01,books,USD,False
9,70262,TXN1533224,P_456,60.00,NaT,toys,USD,True
11,51089,TXN2571945,P_123,45.99,NaT,books,NAN,True
12,77220,TXN4668136,P_456,35.00,2023-04-03,books,USD,True
13,74819,TXN4903402,P_456,35.00,NaT,toys,NAN,False


In [34]:
df.memory_usage(deep=True).sum() / 1024**2  # in MB

np.float64(63.919677734375)

In [37]:
#df.to_parquet('DE/transactions_cleaned.parquet', index=False)
df.to_csv('DE/transactions_cleaned.csv', index=False)